# Scope 1 Carbon Emissions — Combined Forecasting Pipeline
> **Note:** This notebook was built as part of an ESG analytics capstone project.
> Real organizational data has been replaced with placeholder references for portfolio purposes.
> The full pipeline including data is available upon request.

Produces **three sets** of Power BI-ready CSVs in one run:
- **State-based** — one time series per state
- **City-based** — one time series per (state, city)
- **Site-based** — one time series per (state, city, site)

Each granularity exports 4 CSV files:
| File | Description |
|---|---|
|  | Backtest metrics for all models |
|  | Backtest actual vs predicted (long format) |
|  | Historical + best-model future forecast with CIs |
|  | Historical + all-model future forecasts with CIs |


## Imports

In [4]:
import pandas as pd
import numpy as np

from sklearn.impute import KNNImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet


## Configuration
Set your input file and forecast parameters here.

In [5]:
# ── USER SETTINGS ──────────────────────────────────────────────────────────────
INPUT_FILE      = "esg_scope1_data.xlsx"   # path to your Excel file
DATE_CUTOFF     = "2025-03-31"                  # trim data after this date
FORECAST_HORIZON = 36                           # months into the future
BACKTEST_HORIZON = 12                           # months held-out for backtest
MIN_HISTORY      = 24                           # min months needed to model a group
N_KNN_NEIGHBORS  = 3                            # KNN imputation neighbors


## Shared Helpers
These functions are used by all three granularity pipelines.

In [6]:
VALUE_COL = "total_carbon_emissions_kg"

# ── 1. Basic cleaning ──────────────────────────────────────────────────────────
def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[ .+]", "_", regex=True)
        .str.replace(r"^-+",   "", regex=True)
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+",    "_", regex=True)
        .str.strip("_")
    )
    if {"city", "site", "d_end_date"}.issubset(df.columns):
        num_cols     = df.select_dtypes(include=["number"]).columns.tolist()
        non_num_cols = [c for c in df.columns if c not in num_cols]
        df = (
            df.groupby(["city", "site", "d_end_date"], as_index=False)
            .agg({
                **{c: "sum"   for c in num_cols},
                **{c: "first" for c in non_num_cols
                   if c not in ["city", "site", "d_end_date"]}
            })
        )
    if "d_end_date" in df.columns:
        df["d_end_date"] = pd.to_datetime(df["d_end_date"], errors="coerce")
        try:
            df["d_end_date"] = df["d_end_date"].dt.tz_localize(None)
        except Exception:
            pass
        df["date"] = df["d_end_date"].dt.to_period("M").dt.to_timestamp()
    return df


# ── 2. Evaluation metrics ──────────────────────────────────────────────────────
def evaluate(y_true, y_pred):
    mask   = ~(pd.isna(y_true) | pd.isna(y_pred))
    y_true = np.array(y_true)[mask]
    y_pred = np.array(y_pred)[mask]
    if len(y_true) == 0:
        return np.nan, np.nan, np.nan, np.nan
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mu   = np.mean(y_true)
    if mu == 0 or np.isnan(mu):
        return mae, rmse, np.nan, np.nan
    return mae, rmse, (mae / mu) * 100, (rmse / mu) * 100


# ── 3. Fit & forecast a single model ─────────────────────────────────────────
def _fit_forecast(model_name: str, y_train, horizon: int):
    #Return (pred, lower_ci, upper_ci) arrays of length `horizon`
    pred      = np.full(horizon, np.nan)
    lower_ci  = np.full(horizon, np.nan)
    upper_ci  = np.full(horizon, np.nan)
    try:
        if model_name == "Naive":
            pred = np.maximum(np.repeat(y_train.iloc[-1], horizon), 0)

        elif model_name == "ARIMA":
            fc       = ARIMA(y_train, order=(1, 1, 1)).fit().get_forecast(steps=horizon)
            pred     = np.maximum(np.array(fc.predicted_mean), 0)
            ci       = fc.conf_int(alpha=0.05)
            lower_ci = np.maximum(ci.iloc[:, 0].to_numpy(), 0)
            upper_ci = np.maximum(ci.iloc[:, 1].to_numpy(), 0)

        elif model_name == "HoltWinters":
            fitted   = ExponentialSmoothing(y_train, trend="add",
                           seasonal="add", seasonal_periods=12).fit()
            pred     = np.maximum(np.array(fitted.forecast(horizon)), 0)
            if hasattr(fitted, "resid") and fitted.resid is not None:
                std      = np.nanstd(fitted.resid)
                if not np.isnan(std):
                    lower_ci = np.maximum(pred - 1.96 * std, 0)
                    upper_ci = pred + 1.96 * std

        elif model_name == "Prophet":
            train_df = pd.DataFrame({"ds": y_train.index
                                        if hasattr(y_train.index, "dtype")
                                        else range(len(y_train)),
                                     "y": y_train.values})
            # caller must pass prophet_df separately; see _fit_forecast_prophet
            raise NotImplementedError

        elif model_name == "SES":
            pred = np.maximum(np.array(SimpleExpSmoothing(y_train).fit()
                                        .forecast(horizon)), 0)

        elif model_name == "Holt":
            pred = np.maximum(np.array(Holt(y_train).fit().forecast(horizon)), 0)

        elif model_name == "SARIMAX":
            fc       = SARIMAX(y_train, order=(1, 1, 1),
                               seasonal_order=(1, 1, 1, 12),
                               enforce_stationarity=False,
                               enforce_invertibility=False).fit(disp=False)                               .get_forecast(steps=horizon)
            pred     = np.maximum(np.array(fc.predicted_mean), 0)
            ci       = fc.conf_int(alpha=0.05)
            lower_ci = np.maximum(ci.iloc[:, 0].to_numpy(), 0)
            upper_ci = np.maximum(ci.iloc[:, 1].to_numpy(), 0)

    except Exception:
        pass  # leave as NaN
    return pred, lower_ci, upper_ci


def _fit_forecast_prophet(prophet_df: pd.DataFrame, horizon: int):
    #Prophet needs the dates, so it gets its own helper.
    pred     = np.full(horizon, np.nan)
    lower_ci = np.full(horizon, np.nan)
    upper_ci = np.full(horizon, np.nan)
    try:
        m      = Prophet(yearly_seasonality=True,
                         weekly_seasonality=False,
                         daily_seasonality=False)
        m.fit(prophet_df)
        future   = m.make_future_dataframe(periods=horizon, freq="MS")
        fc       = m.predict(future)
        pred     = np.maximum(fc["yhat"].iloc[-horizon:].to_numpy(), 0)
        lower_ci = np.maximum(fc["yhat_lower"].iloc[-horizon:].to_numpy(), 0)
        upper_ci = np.maximum(fc["yhat_upper"].iloc[-horizon:].to_numpy(), 0)
    except Exception:
        pass
    return pred, lower_ci, upper_ci


MODEL_LIST = ["Naive", "ARIMA", "HoltWinters", "Prophet", "SES", "Holt", "SARIMAX"]


## Generic Pipeline Functions
These functions accept a `group_keys` parameter so they work for state, city, or site granularity.

In [7]:
# ── Aggregate monthly ─────────────────────────────────────────────────────────
def aggregate_monthly(df: pd.DataFrame, group_keys: list) -> pd.DataFrame:
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.to_period("M").dt.to_timestamp()
    cols = group_keys + ["date"]
    return df.groupby(cols, as_index=False)[VALUE_COL].sum()


# ── Reindex to full monthly calendar per group ────────────────────────────────
def reindex_monthly(df: pd.DataFrame, group_keys: list) -> pd.DataFrame:
    df   = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    out  = []
    iterable = df.groupby(group_keys) if group_keys else [("__all__", df)]
    for key_vals, g in iterable:
        g = g.sort_values("date")
        if g["date"].isna().all():
            continue
        idx = pd.date_range(g["date"].min(), g["date"].max(), freq="MS")
        g   = g.set_index("date").reindex(idx)
        g.index.name = "date"
        g   = g.reset_index()
        if group_keys:
            vals = key_vals if isinstance(key_vals, tuple) else (key_vals,)
            for k, v in zip(group_keys, vals):
                g[k] = v
        out.append(g)
    return pd.concat(out, ignore_index=True)


# ── KNN imputation per group ──────────────────────────────────────────────────
def knn_impute(df: pd.DataFrame, group_keys: list,
               n_neighbors: int = N_KNN_NEIGHBORS) -> pd.DataFrame:
    df   = df.copy()
    out  = []
    iterable = df.groupby(group_keys) if group_keys else [("__all__", df)]
    for _, g in iterable:
        g = g.sort_values("date").reset_index(drop=True)
        g["_t"] = np.arange(len(g))
        X = g[["_t", VALUE_COL]].values
        g[VALUE_COL] = KNNImputer(n_neighbors=n_neighbors).fit_transform(X)[:, 1]
        g = g.drop(columns=["_t"])
        out.append(g)
    return pd.concat(out, ignore_index=True)


# ── Drop all-zero / all-missing groups ────────────────────────────────────────
def drop_all_zero_or_missing(df: pd.DataFrame, group_keys: list) -> pd.DataFrame:
    df = df.copy()
    if not group_keys:
        total   = len(df)
        bad     = (df[VALUE_COL] == 0).sum() + df[VALUE_COL].isna().sum()
        return df.iloc[0:0].copy() if bad == total else df

    stats = (
        df.groupby(group_keys)[VALUE_COL]
        .agg(total_rows="size",
             zero_count=lambda x: (x == 0).sum(),
             missing_count=lambda x: x.isna().sum())
        .reset_index()
    )
    stats["bad_rows"] = stats["zero_count"] + stats["missing_count"]
    bad = stats.loc[stats["bad_rows"] == stats["total_rows"], group_keys]
    merged = df.merge(bad, on=group_keys, how="left", indicator=True)
    return merged[merged["_merge"] == "left_only"].drop(columns="_merge")


# ── Full preprocessing pipeline ───────────────────────────────────────────────
def preprocess_pipeline(df_raw: pd.DataFrame, group_keys: list) -> pd.DataFrame:
    df = basic_clean(df_raw)
    df = aggregate_monthly(df, group_keys)
    df = reindex_monthly(df, group_keys)
    df = knn_impute(df, group_keys)
    df = drop_all_zero_or_missing(df, group_keys)
    return df


In [8]:
# ── Backtest: all models for all groups ────────────────────────────────────────
def run_backtest(df: pd.DataFrame, group_keys: list,
                 forecast_horizon: int = BACKTEST_HORIZON,
                 min_history: int = MIN_HISTORY):
    df   = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    results, forecast_rows = [], []
    iterable = df.groupby(group_keys) if group_keys else [("__all__", df)]

    for key_vals, group in iterable:
        group = group.sort_values("date").copy()
        if len(group) < min_history:
            continue
        train = group.iloc[:-forecast_horizon].copy()
        test  = group.iloc[-forecast_horizon:].copy()
        if len(train) == 0 or len(test) == 0:
            continue

        y_train     = train[VALUE_COL].reset_index(drop=True)
        y_test      = test[VALUE_COL].reset_index(drop=True)
        test_dates  = test["date"].reset_index(drop=True)
        if y_train.nunique() < 2:
            continue

        key_dict = {}
        if group_keys:
            vals = key_vals if isinstance(key_vals, tuple) else (key_vals,)
            key_dict = dict(zip(group_keys, vals))

        preds = {}
        for m in MODEL_LIST:
            if m == "Prophet":
                prophet_df = train[["date", VALUE_COL]].rename(
                    columns={"date": "ds", VALUE_COL: "y"})
                p, _, _ = _fit_forecast_prophet(prophet_df, len(test))
            else:
                p, _, _ = _fit_forecast(m, y_train, len(test))
            preds[m] = p

        metrics = [(m, *evaluate(y_test, p)) for m, p in preds.items()]
        valid   = [x for x in metrics if pd.notna(x[2])]
        if not valid:
            continue
        best = min(valid, key=lambda x: x[2])[0]

        for m, mae, rmse, mae_pct, rmse_pct in metrics:
            results.append({**key_dict,
                            "model": m, "MAE": mae, "RMSE": rmse,
                            "MAE_pct": mae_pct, "RMSE_pct": rmse_pct,
                            "is_best": int(m == best)})

        for i in range(len(test)):
            base = {**key_dict, "date": test_dates.iloc[i]}
            forecast_rows.append({**base, "series": "Actual",
                                   "value": y_test.iloc[i]})
            for m, p in preds.items():
                forecast_rows.append({**base, "series": m, "value": p[i]})

    return pd.DataFrame(results), pd.DataFrame(forecast_rows)


# ── Future forecast: best model only ─────────────────────────────────────────
def forecast_best_only(df: pd.DataFrame, results_df: pd.DataFrame,
                       group_keys: list, horizon: int = FORECAST_HORIZON):
    df   = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    best_models = (results_df.loc[results_df["is_best"] == 1,
                                  group_keys + ["model"]]
                   .rename(columns={"model": "best_model"})
                   .drop_duplicates())
    output_rows = []
    iterable = df.groupby(group_keys) if group_keys else [("__all__", df)]

    for key_vals, group in iterable:
        group = group.sort_values("date").copy()
        y     = group[VALUE_COL].reset_index(drop=True)
        if len(group) < MIN_HISTORY or y.nunique() < 2:
            continue

        key_dict = {}
        if group_keys:
            vals = key_vals if isinstance(key_vals, tuple) else (key_vals,)
            key_dict = dict(zip(group_keys, vals))

        bm = best_models.copy()
        for k, v in key_dict.items():
            bm = bm[bm[k] == v]
        if bm.empty:
            continue
        best_model = bm["best_model"].iloc[0]

        future_dates = pd.date_range(
            group["date"].max() + pd.offsets.MonthBegin(1),
            periods=horizon, freq="MS")

        if best_model == "Prophet":
            prophet_df = group[["date", VALUE_COL]].rename(
                columns={"date": "ds", VALUE_COL: "y"})
            pred, lower_ci, upper_ci = _fit_forecast_prophet(prophet_df, horizon)
        else:
            pred, lower_ci, upper_ci = _fit_forecast(best_model, y, horizon)

        for i in range(len(group)):
            output_rows.append({**key_dict,
                                 "date": group["date"].iloc[i],
                                 "series": "Historical",
                                 "value": y.iloc[i],
                                 "best_model": best_model,
                                 "lower_ci": np.nan,
                                 "upper_ci": np.nan})
        for i in range(horizon):
            output_rows.append({**key_dict,
                                 "date": future_dates[i],
                                 "series": "Best Forecast",
                                 "value": pred[i],
                                 "best_model": best_model,
                                 "lower_ci": lower_ci[i],
                                 "upper_ci": upper_ci[i]})

    return pd.DataFrame(output_rows)


# ── Future forecast: all models ────────────────────────────────────────────────
def forecast_all(df: pd.DataFrame, group_keys: list,
                 horizon: int = FORECAST_HORIZON):
    df   = df.copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    output_rows = []
    iterable = df.groupby(group_keys) if group_keys else [("__all__", df)]

    for key_vals, group in iterable:
        group = group.sort_values("date").copy()
        y     = group[VALUE_COL].reset_index(drop=True)
        if len(group) < MIN_HISTORY or y.nunique() < 2:
            continue

        key_dict = {}
        if group_keys:
            vals = key_vals if isinstance(key_vals, tuple) else (key_vals,)
            key_dict = dict(zip(group_keys, vals))

        future_dates = pd.date_range(
            group["date"].max() + pd.offsets.MonthBegin(1),
            periods=horizon, freq="MS")

        for model_name in MODEL_LIST:
            if model_name == "Prophet":
                prophet_df = group[["date", VALUE_COL]].rename(
                    columns={"date": "ds", VALUE_COL: "y"})
                pred, lower_ci, upper_ci = _fit_forecast_prophet(prophet_df, horizon)
            else:
                pred, lower_ci, upper_ci = _fit_forecast(model_name, y, horizon)

            for i in range(len(group)):
                output_rows.append({**key_dict,
                                     "date": group["date"].iloc[i],
                                     "model": model_name,
                                     "series": "Historical",
                                     "value": y.iloc[i],
                                     "lower_ci": np.nan,
                                     "upper_ci": np.nan})
            for i in range(horizon):
                output_rows.append({**key_dict,
                                     "date": future_dates[i],
                                     "model": model_name,
                                     "series": "Forecast",
                                     "value": pred[i],
                                     "lower_ci": lower_ci[i],
                                     "upper_ci": upper_ci[i]})

    return pd.DataFrame(output_rows)


## Run All Three Granularities
Runs state → city → site pipelines sequentially and exports all CSVs.

In [10]:
def run_pipeline(df_raw: pd.DataFrame, granularity: str) -> dict[str, pd.DataFrame]:
    """
    granularity : 'state' | 'city' | 'site'
    Returns a dict with keys: model_performance, forecast_long,
                               forecast_best_model, forecast_all_models
    """
    gk_map = {
        "state": ["state"],
        "city":  ["state", "city"],
        "site":  ["state", "city", "site"],
    }
    group_keys = gk_map[granularity]

    print(f"\n{'='*60}")
    print(f"  Running {granularity.upper()}-based pipeline ...")
    print(f"{'='*60}")

    df = preprocess_pipeline(df_raw, group_keys)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["date"] <= pd.Timestamp(DATE_CUTOFF)].copy()

    print(f"  Preprocessed rows : {len(df):,}")

    results_df, forecast_long_df = run_backtest(
        df, group_keys,
        forecast_horizon=BACKTEST_HORIZON,
        min_history=MIN_HISTORY)
    print(f"  Backtest done      : {len(results_df):,} metric rows")

    forecast_best_df = forecast_best_only(df, results_df, group_keys,
                                          horizon=FORECAST_HORIZON)
    print(f"  Best-model forecast: {len(forecast_best_df):,} rows")

    forecast_all_df = forecast_all(df, group_keys, horizon=FORECAST_HORIZON)
    print(f"  All-models forecast: {len(forecast_all_df):,} rows")

    return {
        "model_performance":    results_df,
        "forecast_long":        forecast_long_df,
        "forecast_best_model":  forecast_best_df,
        "forecast_all_models":  forecast_all_df,
    }


In [11]:
# ── MAIN EXECUTION ────────────────────────────────────────────────────────────
df_raw = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df_raw):,} rows from '{INPUT_FILE}'")

outputs = {}
for level in ["state", "city", "site"]:
    outputs[level] = run_pipeline(df_raw, level)

# Export all CSVs
print("\nExporting CSVs ...")
for level, dfs in outputs.items():
    for key, df in dfs.items():
        fname = f"{key}_{level}_based.csv"
        df.to_csv(fname, index=False)
        print(f"  ✓  {fname}  ({len(df):,} rows)")

print("\nAll done. 12 CSV files ready for Power BI.")


Loaded 15,973 rows from 'esg_scope1_data.xlsx'

  Running STATE-based pipeline ...
  Preprocessed rows : 1,025


18:19:50 - cmdstanpy - INFO - Chain [1] start processing
18:19:50 - cmdstanpy - INFO - Chain [1] done processing
18:19:50 - cmdstanpy - INFO - Chain [1] start processing
18:19:50 - cmdstanpy - INFO - Chain [1] done processing
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
18:19:51 - cmdstanpy - INFO - Chain [1] start processing
18:19:51 - cmdstanpy - INFO - Chain [1] done processing
18:19:51 - cmdstanpy - INFO - Chain [1] start processing
18:19:51 - cmdstanpy - INFO - Chain [1] done processing
18:19:51 - cmdstanpy - INFO - Chain [1] start processing
18:19:51 - cmdstanpy - INFO - Chain [1] done processing
18:19:51 - cmdstanpy - INFO - Chain [1] start processing
18:19:52 - cmdstanpy - INFO - Chain [1] done processing
/opt/anaconda3/lib/python3.

  Backtest done      : 70 metric rows


18:19:54 - cmdstanpy - INFO - Chain [1] start processing
18:19:54 - cmdstanpy - INFO - Chain [1] done processing


  Best-model forecast: 1,385 rows


...

  Backtest done      : 238 metric rows


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
18:20:30 - cmdstanpy - INFO - Chain [1] start processing
18:20:30 - cmdstanpy - INFO - Chain [1] done processing
18:20:30 - cmdstanpy - INFO - Chain [1] start processing
18:20:30 - cmdstanpy - INFO - Chain [1] done processing
18:20:30 - cmdstanpy - INFO - Chain [1] start proces

  Best-model forecast: 4,378 rows


18:20:31 - cmdstanpy - INFO - Chain [1] start processing
18:20:31 - cmdstanpy - INFO - Chain [1] done processing
18:20:31 - cmdstanpy - INFO - Chain [1] start processing
18:20:31 - cmdstanpy - INFO - Chain [1] done processing
18:20:31 - cmdstanpy - INFO - Chain [1] start processing
18:20:31 - cmdstanpy - INFO - Chain [1] done processing
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
18:20:32 - cmdstanpy - INFO - Chain [1] start processing
18:20:32 - cmdstanpy - INFO - Chain [1] done processing
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/s

  All-models forecast: 30,646 rows

Exporting CSVs ...
  ✓  model_performance_state_based.csv  (70 rows)
  ✓  forecast_long_state_based.csv  (960 rows)
  ✓  forecast_best_model_state_based.csv  (1,385 rows)
  ✓  forecast_all_models_state_based.csv  (9,695 rows)
  ✓  model_performance_city_based.csv  (105 rows)
  ✓  forecast_long_city_based.csv  (1,440 rows)
  ✓  forecast_best_model_city_based.csv  (2,029 rows)
  ✓  forecast_all_models_city_based.csv  (14,203 rows)
  ✓  model_performance_site_based.csv  (238 rows)
  ✓  forecast_long_site_based.csv  (3,264 rows)
  ✓  forecast_best_model_site_based.csv  (4,378 rows)
  ✓  forecast_all_models_site_based.csv  (30,646 rows)

All done. 12 CSV files ready for Power BI.


## Power BI — How to Load the CSVs

1. **Get Data → Text/CSV** and import all 12 files.
2. Recommended relationships:
   - `model_performance_<level>` ↔ `forecast_best_model_<level>` on `state` / `city` / `site`
3. Key columns for slicers: `state`, `city`, `site`, `model`, `series`, `best_model`
4. Key measures: `value`, `lower_ci`, `upper_ci`, `MAE`, `RMSE`, `is_best`
5. Set `date` column data type to **Date** in Power Query.
